In [67]:
import numpy as np
import json
import re
from konlpy.tag import Okt
from kiwipiepy import Kiwi
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
from wordcloud import WordCloud
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import matplotlib
import warnings
warnings.filterwarnings(action='ignore')
import folium
import seaborn as sns
import requests
from bs4 import BeautifulSoup
import time
from googlesearch import search

In [68]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

import matplotlib.pyplot as plt
import seaborn as sns

In [69]:
df =  pd.read_csv('./12장_data/ns_202104_exam.csv', header=0, engine='python')
print(df.describe) #작업 확인용 출력

<bound method NDFrame.describe of             번호                    도서명                저자    출판사  발행년도  \
0            1                인공지능과 흙            김동훈 지음    민음사  2021   
1            2           가짜 행복 권하는 사회            김태형 지음   갈매나무  2021   
2            3  나도 한 문장 잘 쓰면 바랄 게 없겠네            김선영 지음   블랙피쉬  2021   
3            4                예루살렘 해변  이도 게펜 지음, 임재희 옮김  문학세계사  2021   
4            5  김성곤의 중국한시기행 : 장강·황하 편            김성곤 지음    김영사  2021   
...        ...                    ...               ...    ...   ...   
401677  401678                韓國現代詩大系            채만묵 編著  한국문화사  1996   
401678  401679                  뉴 웨이브        제임스 모나코 지음    한나래  1996   
401679  401680           (최인훈 장편소설)화두            최인훈 지음    민음사  1994   
401680  401681           독일 문학과 세계 문학            吳漢鎭 編著     벽호  1995   
401681  401682             참으로 소중한 생각            김일상 지음    동문사  1995   

               ISBN      세트 ISBN 부가기호    권   주제분류번호  도서권수  대출건수        등록일자  \
0       9.78894E+12   

- [o] 불필요한 열 삭제
- [o] NaN값 처리
  - 완전 없는 값들로 되어있는 **열**은 삭제 가능
- [o] `발행년도` 열의 값을 정규 표현식으로 네 자리 연도로 바꾸기
- [o] 대출 건수 30 초과 항목만 선택
- [o] 중복 행 검색
  - `도서명`, `저자`, `ISBN`을 기준으로 중복된 행이 있는지 찾아보기
- [o] 중복 행 처리
- [o] 같은 도서의 대출건수를 하나로 합치기
- [ ] 값이 누락되었거나 알 수 없는 정보 처리
  - `도서명`, `출판사`, `저자`, `ISBN`, `발행년도`는 크롤링으로 가져와서 처리
  - YES24에서 가져올 수 있으면 가져오기

In [ ]:
df.isnull().sum()

In [ ]:
colum = ['세트 ISBN','부가기호','권', '주제분류번호', '도서권수', '등록일자', 'Unnamed: 13']
df = df.drop(columns=colum)

In [ ]:
df.head(0)

In [ ]:
a = len(df) - df['발행년도'].astype(str).str.match(r'^\d{4}$', na=False).sum()
a

In [ ]:
df['발행년도'] = df['발행년도'].astype(str).str.extract(r'(\d{4})')

In [ ]:
#발행년도 값들을 %y 형태로 바꾸어주었다. 그 후에도 남아있는 값들은 nan값들이다.
df['발행년도'] = pd.to_datetime(df['발행년도'], errors='coerce').dt.strftime('%Y')

In [ ]:
failed_data = df[df['발행년도'].isna()]
print(len(failed_data))

In [ ]:
new_df = df[df['대출건수'] >= 30].copy()
new_df

In [ ]:
new_df.isnull().sum()

In [ ]:
new_df['도서명'] = new_df['도서명'].astype(str).str.strip()
new_df['저자'] = new_df['저자'].astype(str).str.strip()
target_cols = ['도서명', '저자', 'ISBN', '대출건수']
new_df = new_df.drop_duplicates(subset=target_cols, keep='first')
duplicate_count = new_df.duplicated(subset=target_cols).sum()
duplicate_count

In [ ]:
new_df.isnull().sum()

In [ ]:
target_cols = ['도서명', '저자', 'ISBN']
new_df = new_df.drop_duplicates(subset=target_cols, keep='first')
duplicate_count = new_df.duplicated(subset=target_cols).sum()
duplicate_count

In [ ]:
new_df.isnull().sum()

In [ ]:
import google.generativeai as genai
import pandas as pd

In [ ]:
# for index, row in nan_isbn_df.iterrows():
#     query = f"{row['도서명']} {row['저자']} ISBN"
#     results = search(query, num_results=3) # 상위 3개 결과 가져오기
#     # 여기서 결과 링크를 분석하거나 처리...
#     time.sleep(2)

In [ ]:
group_df = new_df.groupby(['도서명', '저자', '출판사', '발행년도', 'ISBN'], as_index=False, dropna=False).agg({
    '대출건수': 'sum'
})
group_df.head(10)

In [ ]:
group_df.isnull().sum()

In [ ]:
nan_rows = group_df[group_df['ISBN'].isna()]
nan_rows.head(10)

In [ ]:
import google.generativeai as genai
genai.configure(api_key="YOUR_GEMINI_API_KEY")
model = genai.GenerativeModel('gemini-2.5-flash')

# 2. ISBN이 NaN인 행만 추출
missing_isbn_mask = group_df['ISBN'].isna()
missing_isbn_df = group_df[missing_isbn_mask].copy()

# 3. 제미나이 API 호출 함수
def get_isbn_from_gemini(row):
    # 책 제목과 저자 정보를 프롬프트에 담습니다.
    prompt = f"""
    아래 도서 정보에 맞는 정확한 ISBN-13을 찾으세요. 
    도서명:'{row['도서명']}', 저자:'{row['저자']}', 발행년도:'{row['발행년도']}'
    
    1. 해당 도서의 ISBN-13을 찾으세요.
    2. 찾은 ISBN으로 실제 도서명과 저자를 확인하세요.
    3. 원본 정보와 일치하면 is_match를 true로, 아니면 false로 하세요.
    
    응답 형식: {{"isbn": "13자리숫자", "is_match": true}}
    결과를 찾지 못하면 {{"isbn": "None", "is_match": false}}
    """
    try:
        #앞에 명령프롬프트 던지고 대답 받아오기
        response = model.generate_content(prompt)
        #대답 앞뒤 공백지우고 저장
        #숫자만 저장
        isbn = response.text.strip()
        return ''.join(filter(str.isdigit, isbn))
    except Exception as e:
        print(f"오류 발생 ({row['도서명']}): {e}")
        return None

# 4. API 실행 (진행 상황을 확인하기 위해 루프 사용)
print(f"보강할 ISBN 개수: {len(missing_isbn_df)}개")
for idx, row in missing_isbn_df.iterrows():
    print(f"검색 중: {row['도서명']}...")
    group_df.at[idx, 'ISBN'] = get_isbn_from_gemini(row)
    time.sleep(1.5)

print("ISBN 보강 작업 완료!")

# 5. 결과 확인
print(group_df[missing_isbn_mask].head())

In [ ]:
print(group_df['ISBN'].isna().sum())

In [ ]:
group_df.isnull().sum()

In [ ]:
def fill_new_df_fast(df):
    cols_to_check = ['도서명', '저자', '출판사', '발행년도']
    #값이 없는 행들 인덱스 저장
    missing_indices = df[df[cols_to_check].isna().any(axis=1)].index
    
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    
    print(f"총 {len(missing_indices)}개의 행 보강 시작...")
    
    for idx in missing_indices:
        isbn = df.loc[idx, 'ISBN']
        if pd.isna(isbn): continue
        
        try:
            search_url = f"https://www.yes24.com/Product/Search?domain=ALL&query={isbn}"
            res = requests.get(search_url, headers=headers)
            soup = BeautifulSoup(res.text, 'html.parser')
            

            item_link = soup.select_one('.gd_name')
            if item_link:
                detail_url = "https://www.yes24.com" + item_link['href']
                
                detail_res = requests.get(detail_url, headers=headers)
                detail_soup = BeautifulSoup(detail_res.text, 'html.parser')
                
                #현재 결측치 찾아보니 출판사와 발행년도만 결측치라서 해당하는 값들만 긁어옴
                # title = detail_soup.select_one('h2.gd_name')
                # author = detail_soup.select_one('.gd_pubArea .gd_auth a')
                pub = detail_soup.select_one('.gd_pub a')
                date = detail_soup.select_one('.gd_date')
                
                # if pd.isna(df.at[idx, '도서명']) and title: df.at[idx, '도서명'] = title.text.strip()
                # if pd.isna(df.at[idx, '저자']) and author: df.at[idx, '저자'] = author.text.strip()
                if pd.isna(df.at[idx, '출판사']) and pub: df.at[idx, '출판사'] = pub.text.strip()
                if pd.isna(df.at[idx, '발행년도']) and date: df.at[idx, '발행년도'] = date.text.strip()
                
            
            time.sleep(1)
            
        except Exception as e:
            print(f"{idx}번 행 크롤링 실패: {e}")
            
    return df

# 실행
group_df = fill_new_df_fast(group_df)

In [ ]:
group_df.isnull().sum()

In [ ]:
nan_group_df = group_df[group_df['발행년도'].isna()].copy()
nan_group_df.head(30)

In [ ]:
#발행년도 값들을 %y 형태로 바꾸어주었다. 그 후에도 남아있는 값들은 nan값들이다.
group_df['발행년도'] = pd.to_datetime(group_df['발행년도'], errors='coerce').dt.strftime('%Y')

In [ ]:
group_df.isnull().sum()

In [ ]:
group_df.to_csv("my_ns.csv", encoding='cp949', index=True)

In [70]:
df_my =  pd.read_csv('./my_ns.csv', encoding='cp949',header=0, engine='python')
print(len(df_my)) #작업 확인용 출력

33576
